# Checkpointing & Early Stopping

**Interview Focus**: What to save, how to resume, gradient checkpointing for memory.

**Key Concepts**:
- Model checkpointing (best + periodic)
- Training resumption (model + optimizer + scheduler + RNG state)
- Early stopping with patience
- Gradient checkpointing (memory vs compute tradeoff)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import os
import tempfile
import copy

## 1. What to Save in a Checkpoint

A complete checkpoint includes:

| Component | Why |
|-----------|-----|
| `model.state_dict()` | Model weights |
| `optimizer.state_dict()` | Momentum buffers, Adam running averages |
| `scheduler.state_dict()` | Current step/epoch for schedule |
| `epoch` / `global_step` | Resume from correct point |
| `best_metric` | Know which model was best |
| `rng_states` | Exact reproducibility |
| `scaler.state_dict()` | AMP grad scaler state |
| `config` | Hyperparameters for reconstruction |

In [ ]:
class CheckpointManager:
    """Manages saving/loading checkpoints."""
    
    def __init__(self, checkpoint_dir, max_checkpoints=5):
        self.checkpoint_dir = checkpoint_dir
        self.max_checkpoints = max_checkpoints
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.saved_checkpoints = []  # (path, metric) pairs
    
    def save_checkpoint(self, state, filename):
        """Save a full training checkpoint."""
        path = os.path.join(self.checkpoint_dir, filename)
        torch.save(state, path)
        return path
    
    def save_best(self, state, metric, filename='best_model.pt'):
        """Save if this is the best metric so far."""
        path = os.path.join(self.checkpoint_dir, filename)
        # Track all saved checkpoints
        self.saved_checkpoints.append((path, metric))
        torch.save(state, path)
        return path
    
    def save_periodic(self, state, epoch, metric=None):
        """Save periodic checkpoint, keeping only max_checkpoints most recent."""
        filename = f'checkpoint_epoch_{epoch}.pt'
        path = self.save_checkpoint(state, filename)
        self.saved_checkpoints.append((path, metric))
        
        # Remove old checkpoints if we exceeded the limit
        while len(self.saved_checkpoints) > self.max_checkpoints:
            old_path, _ = self.saved_checkpoints.pop(0)
            if os.path.exists(old_path) and 'best' not in old_path:
                os.remove(old_path)
        
        return path
    
    @staticmethod
    def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
        """Load a checkpoint and restore all training state."""
        checkpoint = torch.load(path, weights_only=False)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if optimizer and 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        if scheduler and 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        if scaler and 'scaler_state_dict' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        
        # Restore RNG states for reproducibility
        if 'rng_states' in checkpoint:
            torch.set_rng_state(checkpoint['rng_states']['torch'])
            np.random.set_state(checkpoint['rng_states']['numpy'])
        
        return checkpoint.get('epoch', 0), checkpoint.get('best_metric', None)


def create_checkpoint_state(model, optimizer, scheduler, epoch, global_step,
                             best_metric, config=None, scaler=None):
    """Create a complete checkpoint dictionary."""
    state = {
        'epoch': epoch,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_metric': best_metric,
        'rng_states': {
            'torch': torch.get_rng_state(),
            'numpy': np.random.get_state(),
        },
        'config': config or {},
    }
    if scheduler:
        state['scheduler_state_dict'] = scheduler.state_dict()
    if scaler:
        state['scaler_state_dict'] = scaler.state_dict()
    return state


print("CheckpointManager implemented.")

## 2. Early Stopping

In [ ]:
class EarlyStopping:
    """Early stopping to halt training when validation metric stops improving.
    
    Args:
        patience: Number of epochs to wait for improvement
        min_delta: Minimum change to qualify as an improvement
        mode: 'min' for loss, 'max' for accuracy/F1
    """
    
    def __init__(self, patience=5, min_delta=0.0, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_metric = None
        self.counter = 0
        self.should_stop = False
    
    def _is_improvement(self, current, best):
        if self.mode == 'min':
            return current < best - self.min_delta
        else:
            return current > best + self.min_delta
    
    def __call__(self, metric):
        """Call after each validation. Returns True if should stop."""
        if self.best_metric is None:
            self.best_metric = metric
            return False
        
        if self._is_improvement(metric, self.best_metric):
            self.best_metric = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                return True
        
        return False


# Demo
stopper = EarlyStopping(patience=3, mode='min')
val_losses = [0.5, 0.45, 0.43, 0.44, 0.45, 0.46, 0.47]  # stops improving after epoch 3

for epoch, loss in enumerate(val_losses):
    should_stop = stopper(loss)
    status = 'STOP' if should_stop else f'patience={stopper.counter}/{stopper.patience}'
    print(f"Epoch {epoch}: val_loss={loss:.4f}, best={stopper.best_metric:.4f}, {status}")
    if should_stop:
        break

## 3. Gradient Checkpointing (Activation Checkpointing)

**Problem**: During backprop, PyTorch stores all intermediate activations. For deep models, this dominates memory.

**Solution**: Don't store all activations. During backward pass, recompute them on the fly.

**Tradeoff**: ~30% less memory, ~20-30% more compute (one extra forward pass per segment).

For a model with L layers:
- Without checkpointing: O(L) activation memory
- With checkpointing every sqrt(L) layers: O(sqrt(L)) memory, 1 extra forward pass

In [ ]:
from torch.utils.checkpoint import checkpoint as torch_checkpoint

class TransformerBlockNoCheckpoint(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class TransformerWithGradientCheckpointing(nn.Module):
    """Transformer using gradient checkpointing to save memory."""
    
    def __init__(self, n_layers=12, d_model=256):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlockNoCheckpoint(d_model) for _ in range(n_layers)
        ])
        self.use_checkpoint = True
    
    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint and self.training:
                # Activations won't be stored; recomputed during backward
                x = torch_checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x


# Compare memory (conceptual — exact numbers depend on GPU)
n_layers = 12
d_model = 256
batch_size = 8
seq_len = 128

# Without checkpointing: store activations for all 12 layers
activation_per_layer = batch_size * seq_len * d_model * 4  # bytes (FP32)
total_without = activation_per_layer * n_layers

# With checkpointing: only store at checkpoint boundaries
# If we checkpoint every layer: store only 1 layer's activations at a time
total_with = activation_per_layer * 1  # (simplified)

print(f"Activation memory estimate (FP32):")
print(f"  Without checkpointing: {total_without / 1e6:.1f} MB ({n_layers} layers stored)")
print(f"  With checkpointing:    {total_with / 1e6:.1f} MB (recomputed on the fly)")
print(f"  Savings:               {(1 - total_with/total_without)*100:.0f}%")
print(f"  Compute overhead:      ~{33}% (extra forward passes during backward)")

# Verify it works
model_gc = TransformerWithGradientCheckpointing(n_layers=4, d_model=64)
x = torch.randn(2, 16, 64)
out = model_gc(x)
loss = out.sum()
loss.backward()
print(f"\nGradient checkpointing test: output shape={out.shape}, backward pass OK.")

## 4. Complete Training Loop with Everything

In [ ]:
# Create synthetic dataset
torch.manual_seed(42)
n_samples = 800
X = torch.randn(n_samples, 20)
w_true = torch.randn(20, 1)
y = (X @ w_true + 0.3 * torch.randn(n_samples, 1)).squeeze()
y_cls = (y > y.median()).long()

# 80/20 split
n_train = int(0.8 * n_samples)
X_train, X_val = X[:n_train], X[n_train:]
y_train, y_val = y_cls[:n_train], y_cls[n_train:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
def full_training_loop(model, train_loader, val_loader, optimizer, scheduler,
                       criterion, checkpoint_dir, max_epochs=50, patience=7,
                       save_every=5, resume_from=None):
    """Complete training loop with checkpointing, early stopping, and resumption."""
    
    ckpt_mgr = CheckpointManager(checkpoint_dir, max_checkpoints=3)
    early_stopper = EarlyStopping(patience=patience, mode='min')
    
    start_epoch = 0
    best_val_loss = float('inf')
    global_step = 0
    train_losses, val_losses, val_accs = [], [], []
    
    # Resume from checkpoint if provided
    if resume_from and os.path.exists(resume_from):
        start_epoch, best_val_loss = CheckpointManager.load_checkpoint(
            resume_from, model, optimizer, scheduler
        )
        print(f"Resumed from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")
    
    for epoch in range(start_epoch, max_epochs):
        # === Training ===
        model.train()
        epoch_train_loss = 0
        n_batches = 0
        
        for inputs, targets in train_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            if scheduler:
                scheduler.step()
            
            epoch_train_loss += loss.item()
            n_batches += 1
            global_step += 1
        
        avg_train_loss = epoch_train_loss / n_batches
        train_losses.append(avg_train_loss)
        
        # === Validation ===
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item()
                correct += (outputs.argmax(1) == targets).sum().item()
                total += len(targets)
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        val_losses.append(avg_val_loss)
        val_accs.append(val_acc)
        
        # === Checkpointing ===
        state = create_checkpoint_state(
            model, optimizer, scheduler, epoch + 1, global_step, best_val_loss
        )
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            state['best_metric'] = best_val_loss
            ckpt_mgr.save_best(state, best_val_loss)
            marker = ' *best*'
        else:
            marker = ''
        
        # Save periodic checkpoint
        if (epoch + 1) % save_every == 0:
            ckpt_mgr.save_periodic(state, epoch + 1)
        
        print(f"Epoch {epoch+1:3d}: train_loss={avg_train_loss:.4f}, "
              f"val_loss={avg_val_loss:.4f}, val_acc={val_acc:.4f}{marker}")
        
        # === Early Stopping ===
        if early_stopper(avg_val_loss):
            print(f"\nEarly stopping at epoch {epoch+1} (patience={patience})")
            break
    
    return train_losses, val_losses, val_accs


print("Full training loop defined.")

In [ ]:
# Run training
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 2)
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50 * len(train_loader))
criterion = nn.CrossEntropyLoss()

checkpoint_dir = tempfile.mkdtemp(prefix='checkpoints_')
print(f"Checkpoint dir: {checkpoint_dir}\n")

train_losses, val_losses, val_accs = full_training_loop(
    model, train_loader, val_loader, optimizer, scheduler, criterion,
    checkpoint_dir=checkpoint_dir,
    max_epochs=50,
    patience=7,
    save_every=10
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mark the best epoch
best_epoch = np.argmin(val_losses)
axes[0].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best epoch: {best_epoch+1}')
axes[0].legend()

axes[1].plot(val_accs, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best val loss: {min(val_losses):.4f} at epoch {best_epoch + 1}")

## 5. Resume from Checkpoint Demo

In [ ]:
# Demonstrate checkpoint resumption
print("Saved checkpoints:")
for f in sorted(os.listdir(checkpoint_dir)):
    fpath = os.path.join(checkpoint_dir, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

# Load best model
best_path = os.path.join(checkpoint_dir, 'best_model.pt')
if os.path.exists(best_path):
    new_model = nn.Sequential(
        nn.Linear(20, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 2)
    )
    new_optimizer = torch.optim.Adam(new_model.parameters(), lr=1e-3)
    
    epoch, best_metric = CheckpointManager.load_checkpoint(
        best_path, new_model, new_optimizer
    )
    print(f"\nLoaded best model from epoch {epoch}, best_metric={best_metric:.4f}")
    
    # Verify loaded model matches
    new_model.eval()
    with torch.no_grad():
        preds = new_model(X_val).argmax(1)
        acc = (preds == y_val).float().mean().item()
    print(f"Loaded model val accuracy: {acc:.4f}")

## 6. Gradient Checkpointing Memory Analysis

In [ ]:
# Visualize gradient checkpointing tradeoff
n_layers_range = np.arange(1, 49)

# Memory per layer = batch_size * seq_len * hidden_dim * dtype_size
# Without checkpointing: all L layers stored
# With checkpointing every k layers: L/k segments stored + recompute within segment
# Optimal k = sqrt(L)

mem_without = n_layers_range  # O(L)
mem_with_sqrt = np.sqrt(n_layers_range) * 2  # O(sqrt(L)) approximately
mem_with_every = np.ones_like(n_layers_range, dtype=float) * 2  # checkpoint every layer

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(n_layers_range, mem_without, 'b-', linewidth=2, label='No checkpointing')
axes[0].plot(n_layers_range, mem_with_sqrt, 'r--', linewidth=2, label='Checkpoint every sqrt(L)')
axes[0].plot(n_layers_range, mem_with_every, 'g:', linewidth=2, label='Checkpoint every layer')
axes[0].set_xlabel('Number of Layers')
axes[0].set_ylabel('Activation Memory (relative)')
axes[0].set_title('Activation Memory vs Layers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compute overhead
compute_without = np.ones_like(n_layers_range, dtype=float)  # 1x
compute_with = np.ones_like(n_layers_range, dtype=float) * 1.33  # ~33% overhead

axes[1].bar(['No Checkpointing', 'With Checkpointing'], [1.0, 1.33], 
            color=['#2196F3', '#FF9800'])
axes[1].set_ylabel('Relative Compute Time')
axes[1].set_title('Compute Overhead')
axes[1].set_ylim(0, 1.6)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate([1.0, 1.33]):
    axes[1].text(i, v + 0.03, f'{v:.2f}x', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Interview Questions & Answers

---

**Q: What should you save in a checkpoint?**

Everything needed to resume training exactly:
- `model.state_dict()` — model weights
- `optimizer.state_dict()` — momentum, Adam states (losing these = like restarting training)
- `scheduler.state_dict()` — LR schedule position
- `epoch` / `global_step` — where to resume
- `best_metric` — for early stopping / best model tracking
- `rng_states` — for exact reproducibility
- `scaler.state_dict()` — for AMP loss scaling
- `config` — so you can reconstruct the model architecture

Common mistake: Only saving model weights. If you lose optimizer state (Adam's running mean/variance), resuming training will cause a spike in loss.

---

**Q: Gradient checkpointing — memory vs compute tradeoff?**

- **Memory**: Reduces activation memory from O(L) to O(sqrt(L)) by not storing intermediate activations
- **Compute**: ~30% overhead because each segment's activations are recomputed during backward pass (effectively an extra forward pass per segment)
- **When to use**: Model almost fits in GPU memory, or you want a larger batch size
- **In PyTorch**: `torch.utils.checkpoint.checkpoint(fn, *args)` or `model.gradient_checkpointing_enable()` in HuggingFace

---

**Q: Early stopping — how to choose patience?**

- Too small: Stop before the model converges (underfitting)
- Too large: Waste compute on overfitting epochs
- Rule of thumb: 5-10 epochs for smaller models, 3-5 for expensive models
- Always keep the best checkpoint, not the last one

---

**Q: `model.state_dict()` vs `model`?**

- `torch.save(model.state_dict(), path)` — saves only weights (recommended: portable, smaller)
- `torch.save(model, path)` — pickles the entire model (fragile: breaks if code changes)
- Always use `state_dict()` in production

## Quick Reference

```python
# Save checkpoint
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_val_loss': best_val_loss,
}, 'checkpoint.pt')

# Load checkpoint
ckpt = torch.load('checkpoint.pt', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])

# Gradient checkpointing (HuggingFace)
model.gradient_checkpointing_enable()

# Gradient checkpointing (PyTorch)
from torch.utils.checkpoint import checkpoint
output = checkpoint(layer, input, use_reentrant=False)
```